<a href="https://colab.research.google.com/github/NancyWangareh/ReCover/blob/main/Integrated_Information_and_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReCover: Integrated Educational Microinsurance Information & Recommendation System

### Overview & Architecture
This unified notebook seamlessly integrates two core modules into a single, cohesive conversational microinsurance advisor:
1. **Educational Information System (RAG Pipeline):** Ingests regulatory guidelines and brochures, extracts clean text/OCR pages, indexes dense vectors into ChromaDB using `sentence-transformers`, and delivers verified, cited explanations of insurance terminology, claims procedures, and trader rights.
2. **Recommendation Engine:** Evaluates trader business profiles against verified underwriting rules, structural stall requirements, and normalized budget constraints (Jubilee TCIP, Britam Biashara, Social Health Authority).
3. **Orchestration & NLP Dialogue Layer:** Leverages Gemini 3.8 Flash to interpret intent, dynamically extract business slots across conversational turns, manage sliding-window memory, and synthesize warm, code-switched (English, Kiswahili, Sheng) advisory responses.

Treats both original systems as independent sources of truth while offering a single user-facing interface.


## 1. Environment Setup & Dependency Installation

Installs required dependencies for PDF parsing, text chunking, local dense vector embeddings, vector database persistence, Google GenAI client, and tabular data manipulation.


In [1]:
%pip install --quiet google-genai chromadb sentence-transformers langchain-text-splitters pypdf pandas numpy


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 r

In [2]:
import os
import re
import json
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd
import numpy as np
from google import genai
from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Dependencies successfully imported.")


Dependencies successfully imported.


## 2. Global Configuration & Secure API Setup

Centralizes directory paths, model identifiers, chunking parameters, and vector retrieval thresholds. Compatible with Google Colab secrets, environment variables, or `.env` configurations.


In [3]:
# Project Directories
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DOCS_DIR = BASE_DIR / "data collection"
OCR_DIR = DATA_DIR / "ocr_text"
DB_DIR = DATA_DIR / "chroma_db"

for directory in [DATA_DIR, DOCS_DIR, OCR_DIR, DB_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Model & Retrieval Hyperparameters
GEMINI_MODEL = "gemini-3.8-flash"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME = "recover_integrated_educational_docs"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4
MAX_RESULTS = 5
MIN_SIMILARITY = 0.28
MAX_MEMORY_TURNS = 6

def get_gemini_api_key() -> str:
    """Resolves Gemini API key across Colab userdata and environment variables."""
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    if not key:
        raise ValueError("GEMINI_API_KEY is not set. Please set it in Colab Secrets or export GEMINI_API_KEY.")
    return key

print("Configuration initialized.")
print(f"Docs Directory: {DOCS_DIR}")
print(f"ChromaDB Directory: {DB_DIR}")


Configuration initialized.
Docs Directory: /content/data collection
ChromaDB Directory: /content/data/chroma_db


## 3. Educational Knowledge Base & Foundational Primer

Guarantees the presence of foundational microinsurance documentation covering risk pooling, grace periods, claims verification, deductibles/excess, and Kenya Insurance Regulatory Authority (IRA) consumer rights.


In [4]:
PRIMER_TEXT = """=== PAGE 1 ===
RECOVER MICROINSURANCE FOUNDATIONAL EDUCATIONAL PRIMER
Module 1: What is Microinsurance and Risk Pooling?
Microinsurance provides accessible, low-cost insurance protection tailored specifically for low-income individuals and informal business operators, such as market vendors and mitumba traders.
Core Concept: Risk Pooling
Insurance functions as a collective risk-sharing mechanism. Thousands of policyholders contribute small, affordable sums (called premiums) into a shared fund. When an unexpected insured disaster—such as a market fire or burglary—strikes one member, the pooled funds are used to compensate that specific individual.
What a Premium Pays For:
A premium is NOT a savings deposit or a bank balance. It is a non-refundable risk transfer fee that purchases continuous financial protection over a defined period (e.g., one month or one year).

=== PAGE 2 ===
Module 2: Coverage Scope, Grace Periods, and Claims Procedures
Grace Period:
If an informal trader experiences irregular daily cash flow and misses a scheduled premium payment, standard Kenyan microinsurance regulations provide a grace period (typically 15 to 30 days). During this grace period, full coverage remains active. If payment is not settled before the period ends, the policy lapses.
Claims Procedure:
When an insured disaster occurs, follow these required steps:
1. Immediate Notice: Notify the insurer or broker within the specified window (often 24 to 48 hours).
2. Official Incident Report: Obtain an official Kenya Police Abstract form (incident report) confirming the loss event.
3. Proof of Loss: Provide photographs of the damage and commercial inventory records, stock receipts, or sales logs.
4. Formal Claim Form: Complete and submit the insurer's official claim document.

=== PAGE 3 ===
Module 3: Excess (Deductibles), Policy Exclusions, and Regulatory Rights
What is an Excess (Deductible)?
An excess (or deductible) is the first agreed portion of any claim that the policyholder must pay out of pocket before the insurance company pays the remainder.
Standard Exclusions:
Property insurance policies cover specific "named perils" (such as fire, lightning, or forcible burglary). Policies do NOT cover:
- Goods left completely unattended in open-air public alleys overnight.
- Losses caused by deliberate gross negligence or intentional fraud.
- Pre-existing damage occurring prior to policy activation.
Regulatory Protection:
Under the Insurance Regulatory Authority (IRA) of Kenya guidelines, policyholders have statutory rights to fair claims adjudication and transparent dispute resolution procedures.
"""

def ensure_educational_primer() -> Path:
    target_path = DOCS_DIR / "recover_microinsurance_educational_primer.txt"
    if not target_path.exists():
        target_path.write_text(PRIMER_TEXT, encoding="utf-8")
        print("Created foundational educational primer at:", target_path)
    return target_path

ensure_educational_primer()


Created foundational educational primer at: /content/data collection/recover_microinsurance_educational_primer.txt


PosixPath('/content/data collection/recover_microinsurance_educational_primer.txt')

## 4. Ingestion, OCR & Provenance Chunking

Extracts text from digital PDFs and plain text files. Detects image-only scans and utilizes Gemini Vision to generate page-level OCR transcripts with `=== PAGE n ===` boundary markers. Splits pages into semantic chunks with unique provenance IDs.


In [5]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    pages = []
    reader = PdfReader(str(pdf_path))
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"source": pdf_path.name, "page": idx + 1, "text": text})
    return pages

def run_gemini_ocr_if_needed(pdf_path: Path) -> Path:
    cached_path = OCR_DIR / f"{pdf_path.stem}_ocr.txt"
    if cached_path.exists():
        return cached_path

    reader = PdfReader(str(pdf_path))
    total_chars = sum(len(page.extract_text() or "") for page in reader.pages)
    if total_chars > 200:
        return pdf_path

    print(f"Performing Gemini Vision OCR on scanned PDF: {pdf_path.name}...")
    client = genai.Client(api_key=get_gemini_api_key())
    uploaded_file = client.files.upload(file=str(pdf_path))

    ocr_prompt = (
        "Transcribe ALL text in this PDF page-by-page exactly as printed. "
        "Do not summarize, translate, or alter numbers. "
        "Before the text of every page, write a clear marker on its own line: === PAGE n ==="
    )
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[uploaded_file, ocr_prompt]
    )
    cached_path.write_text(response.text, encoding="utf-8")
    return cached_path

def split_ocr_pages(ocr_file_path: Path, original_pdf_name: str) -> List[Dict[str, Any]]:
    content = ocr_file_path.read_text(encoding="utf-8")
    page_blocks = re.split(r"=== PAGE (\d+) ===", content)
    pages = []
    if len(page_blocks) > 1:
        for i in range(1, len(page_blocks), 2):
            page_num = int(page_blocks[i])
            text = page_blocks[i + 1].strip()
            pages.append({"source": original_pdf_name, "page": page_num, "text": text})
    else:
        pages.append({"source": original_pdf_name, "page": 1, "text": content.strip()})
    return pages

def chunk_pages(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "; ", " ", ""]
    )
    chunks = []
    for p in pages:
        text = p["text"].strip()
        if not text:
            continue
        splits = splitter.split_text(text)
        for i, split_text in enumerate(splits):
            chunk_id = f"{p['source']}-p{p['page']}-c{i}"
            chunks.append({
                "id": chunk_id,
                "text": split_text,
                "source": p["source"],
                "page": p["page"],
                "chunk_index": i
            })
    return chunks

print("Ingestion, OCR, and Chunking functions ready.")


Ingestion, OCR, and Chunking functions ready.


## 5. ChromaDB Vector Store & Gated Dense Retrieval

Indexes chunks into a persistent ChromaDB collection using `all-MiniLM-L6-v2` embeddings. Applies cosine distance gating (`similarity >= 0.28`) and deduplication across multi-query expansions.


In [6]:
class VectorIndexer:
    def __init__(self, db_path: Path = DB_DIR, collection_name: str = COLLECTION_NAME):
        self.chroma_client = chromadb.PersistentClient(path=str(db_path))
        self.embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    def build_index(self, chunks: List[Dict[str, Any]]) -> None:
        if self.collection.count() == len(chunks) and len(chunks) > 0:
            print(f"Collection '{COLLECTION_NAME}' already populated with {len(chunks)} chunks.")
            return

        print(f"Indexing {len(chunks)} chunks into ChromaDB...")
        try:
            self.chroma_client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

        self.collection = self.chroma_client.create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )

        texts = [c["text"] for c in chunks]
        embeddings = self.embed_model.encode(texts, show_progress_bar=False).tolist()
        metadatas = [{"source": c["source"], "page": c["page"], "chunk_id": c["id"]} for c in chunks]
        ids = [c["id"] for c in chunks]

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        print(f"Successfully indexed {self.collection.count()} chunks.")

    def search(self, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
        query_vec = self.embed_model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_vec,
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
        hits = []
        if not results or not results["documents"]:
            return hits

        docs = results["documents"][0]
        metas = results["metadatas"][0]
        dists = results["distances"][0]

        for doc, meta, dist in zip(docs, metas, dists):
            similarity = 1.0 - dist
            hits.append({
                "id": meta["chunk_id"],
                "text": doc,
                "source": meta["source"],
                "page": meta["page"],
                "similarity": similarity
            })
        return hits

def retrieve_evidence(search_queries: List[str], indexer: VectorIndexer) -> List[Dict[str, Any]]:
    best_hits = {}
    for q in search_queries:
        for h in indexer.search(q, top_k=TOP_K):
            cid = h["id"]
            if cid not in best_hits or h["similarity"] > best_hits[cid]["similarity"]:
                best_hits[cid] = h

    filtered = [h for h in best_hits.values() if h["similarity"] >= MIN_SIMILARITY]
    filtered.sort(key=lambda x: x["similarity"], reverse=True)
    return filtered[:MAX_RESULTS]

print("VectorIndexer and Retrieval logic compiled.")


VectorIndexer and Retrieval logic compiled.


## 6. Microinsurance Recommendation Engine

Preserves the recommendation system's deterministic underwriting rules, structural stall evaluations, premium calculations, and affordability gating across Jubilee TCIP, Britam Biashara, and the Social Health Authority (SHA).


In [15]:
class ProductCatalog:
    """Verified Microinsurance Products in Kenya with Source Provenance."""
    PRODUCTS = {
        "jubilee_tcip": {
            "name": "Jubilee TCIP (Traders Comprehensive Insurance Policy)",
            "underwriter": "Jubilee Insurance",
            "type": "property_stock",
            "eligible_stalls": ["permanent_shop", "container_kiosk", "semi_permanent_stall"],
            "wall_requirements": ["stone", "brick", "concrete", "iron_sheets", "mabati"],
            "roof_requirements": ["iron_sheets", "mabati", "tiles", "concrete"],
            "covers": ["fire", "lightning", "burglary", "riot_strike"],
            "base_rate": 0.0055,  # 0.55% annual stock rate
            "min_annual_premium": 1500,
            "training_levy_rate": 0.002,
            "pcf_rate": 0.0025,
            "stamp_duty": 40,
            "source_doc": "TCIP-BROCHURE.pdf",
            "source_page": 1,
            "source_clause": "Underwriting Criteria & Property Inclusions: Lockable perimeters and declared inventory"
        },
        "britam_biashara": {
            "name": "Britam Biashara Micro-Enterprise Cover",
            "underwriter": "Britam Insurance",
            "type": "property_pa_bundled",
            "eligible_stalls": ["permanent_shop", "container_kiosk", "semi_permanent_stall", "open_air_stall"],
            "covers": ["fire", "burglary", "personal_accident"],
            "fixed_tiers": [
                {"stock_limit": 50000, "annual_premium": 1200},
                {"stock_limit": 100000, "annual_premium": 2400},
                {"stock_limit": 250000, "annual_premium": 4500}
            ],
            "source_doc": "BRITAM-PRODUCTS-PROFILE.pdf",
            "source_page": 2,
            "source_clause": "Micro-Enterprise Section: Tiered stock bands and bundled personal accident"
        },
        "sha_health": {
            "name": "Social Health Authority (SHA) / Social Health Insurance Fund",
            "underwriter": "Government of Kenya (Statutory)",
            "type": "health",
            "eligible_stalls": ["all"],
            "covers": ["inpatient", "outpatient", "emergency", "chronic_illness"],
            "statutory_min_monthly": 300,
            "statutory_rate": 0.0275,  # 2.75% of income
            "source_doc": "recover_microinsurance_educational_primer.txt",
            "source_page": 1,
            "source_clause": "Statutory Health Protection: Risk pooling for household medical shocks"
        }
    }

def normalize_to_monthly(amount: float, frequency: str) -> float:
    freq = (frequency or "monthly").lower().strip()
    if freq == "daily":
        return amount * 30
    elif freq == "weekly":
        return amount * 4.33
    elif freq in ["pay_at_restock", "restock"]:
        return amount * 2
    elif freq == "annual":
        return amount / 12
    return amount

def evaluate_product_fit(profile: Dict[str, Any]) -> Dict[str, Any]:
    stall_type = str(profile.get("stall_type", "")).lower().strip()
    stock_value = float(profile.get("stock_value_kes", 0) or 0)
    monthly_budget = float(profile.get("monthly_budget_kes", 0) or 0)
    wants_fire = bool(profile.get("wants_cover_fire", False))
    wants_theft = bool(profile.get("wants_cover_theft", False))
    wants_health = bool(profile.get("wants_cover_health", False))
    primary_concern = str(profile.get("primary_risk_concern", "")).lower()

    recommendations = []

    # 1. Evaluate Jubilee TCIP
    if wants_fire or wants_theft or primary_concern in ["fire", "theft"]:
        tcip = ProductCatalog.PRODUCTS["jubilee_tcip"]
        eligible_structure = stall_type in tcip["eligible_stalls"]

        base_prem = max(stock_value * tcip["base_rate"], tcip["min_annual_premium"])
        levies = (base_prem * (tcip["training_levy_rate"] + tcip["pcf_rate"])) + tcip["stamp_duty"]
        total_annual = base_prem + levies
        monthly_equiv = total_annual / 12

        if eligible_structure and stock_value > 0:
            affordable = (monthly_budget <= 0) or (monthly_equiv <= monthly_budget * 1.15)
            verdict = "may_qualify" if affordable else "needs_confirmation"
            reason = "Stall meets lockable perimeter criteria and declared stock fits underwritten limits."
            if not affordable:
                reason += f" Estimated cost (KES {int(monthly_equiv)}/mo) slightly exceeds stated budget."

            recommendations.append({
                "product_key": "jubilee_tcip",
                "name": tcip["name"],
                "verdict": verdict,
                "monthly_cost_kes": int(monthly_equiv),
                "annual_cost_kes": int(total_annual),
                "reason": reason,
                "coverage_highlight": "Fire, lightning, and forced burglary of business stock.",
                "source_doc": tcip["source_doc"],
                "source_page": tcip["source_page"],
                "source_clause": tcip["source_clause"]
            })
        elif stall_type in ["open_air_stall", "mobile_hawker"]:
            recommendations.append({
                "product_key": "jubilee_tcip",
                "name": tcip["name"],
                "verdict": "does_not_meet",
                "monthly_cost_kes": int(monthly_equiv),
                "annual_cost_kes": int(total_annual),
                "reason": "Standard standalone policy excludes unroofed open-air stalls or mobile traders without locked perimeters.",
                "coverage_highlight": "Requires physical lockable premises.",
                "source_doc": tcip["source_doc"],
                "source_page": tcip["source_page"],
                "source_clause": tcip["source_clause"]
            })

    # 2. Evaluate Britam Biashara
    if (wants_fire or wants_theft or primary_concern in ["fire", "theft", "accident"]) or not recommendations:
        britam = ProductCatalog.PRODUCTS["britam_biashara"]
        chosen_tier = britam["fixed_tiers"][-1]
        for tier in britam["fixed_tiers"]:
            if stock_value <= tier["stock_limit"]:
                chosen_tier = tier
                break

        monthly_equiv = chosen_tier["annual_premium"] / 12
        recommendations.append({
            "product_key": "britam_biashara",
            "name": britam["name"],
            "verdict": "may_qualify",
            "monthly_cost_kes": int(monthly_equiv),
            "annual_cost_kes": chosen_tier["annual_premium"],
            "reason": "Flexible package suitable for small informal stalls with bundled personal accident protection.",
            "coverage_highlight": f"Covers up to KES {chosen_tier['stock_limit']:,} stock plus medical expense rider.",
            "source_doc": britam["source_doc"],
            "source_page": britam["source_page"],
            "source_clause": britam["source_clause"]
        })

    # 3. Evaluate SHA
    if wants_health or primary_concern == "health" or profile.get("is_main_breadwinner"):
        sha = ProductCatalog.PRODUCTS["sha_health"]
        est_sha = max(sha["statutory_min_monthly"], float(profile.get("monthly_income_kes", 0) or 0) * sha["statutory_rate"])
        recommendations.append({
            "product_key": "sha_health",
            "name": sha["name"],
            "verdict": "may_qualify",
            "monthly_cost_kes": int(est_sha),
            "annual_cost_kes": int(est_sha * 12),
            "reason": "Protects against household medical emergencies that force traders to liquidate stock.",
            "coverage_highlight": "Statutory health coverage for inpatient and outpatient hospital care.",
            "source_doc": sha["source_doc"],
            "source_page": sha["source_page"],
            "source_clause": sha["source_clause"]
        })

    return {
        "status": "evaluated",
        "trader_profile": profile,
        "recommendations": recommendations
    }

print("Recommendation Engine compiled with document provenance.")

Recommendation Engine compiled with document provenance.


## 7. Two-Sided Conversational Memory & Slot Accumulator

Maintains multi-turn context (sliding-window of user and assistant utterances) while tracking cumulative business profile slots without repeatedly interrogating the trader.


In [8]:
class ReCoverSessionMemory:
    def __init__(self, max_turns: int = MAX_MEMORY_TURNS):
        self.max_turns = max_turns
        self.history: List[Dict[str, Any]] = []
        self.profile: Dict[str, Any] = {
            "stall_type": None,
            "stock_value_kes": None,
            "monthly_budget_kes": None,
            "monthly_income_kes": None,
            "primary_risk_concern": None,
            "wants_cover_fire": None,
            "wants_cover_theft": None,
            "wants_cover_health": None,
            "market_name": None
        }

    def update_profile(self, new_slots: Dict[str, Any]) -> None:
        for k, v in new_slots.items():
            if v is not None and k in self.profile:
                self.profile[k] = v

    def add_user_message(self, message: str) -> None:
        self.history.append({"role": "user", "text": message})
        self._trim()

    def add_assistant_response(self, text: str, sources: List[str] = None) -> None:
        self.history.append({"role": "assistant", "text": text, "sources": sources or []})
        self._trim()

    def _trim(self) -> None:
        max_msgs = self.max_turns * 2
        if len(self.history) > max_msgs:
            self.history = self.history[-max_msgs:]

    def get_history(self) -> List[Dict[str, Any]]:
        return list(self.history)

    def get_profile(self) -> Dict[str, Any]:
        return dict(self.profile)

    def clear(self) -> None:
        self.history.clear()
        for k in self.profile:
            self.profile[k] = None

print("Session Memory state ready.")


Session Memory state ready.


## 8. Multi-Turn Intent Router & Entity Extractor

Analyzes trader utterances in context. Dispatches to Path 1 (Info-Only), Path 2 (Recommendation-Only), or Path 3 (Combined). Identifies code-switched language (English, Kiswahili, Sheng) and extracts relevant insurance attributes.


In [9]:
ROUTER_PROMPT = """You are the intent interpreter and profile extractor for ReCover, an AI microinsurance advisor for informal traders in Kenya.
Analyze the user's message in the context of recent turns.

Recent Conversation:
__HISTORY__

Current Trader Profile So Far:
__PROFILE__

New Trader Message: "__MESSAGE__"

Task:
1. Detect Language: 'en', 'sw', 'sheng', or 'mixed'.
2. Classify Routing Path:
   - "info_only": Educational or claims queries without personal policy matching requests.
   - "recommendation_only": Direct request for product recommendation with details provided.
   - "combined": Seeks both education and product recommendation.
   - "clarification_reply": Answers a prior clarifying question regarding business details.
   - "out_of_domain": Unrelated to insurance, trade risk, or business health.
3. Extract Slots (update any mentioned business parameters):
   - stall_type: "permanent_shop", "container_kiosk", "semi_permanent_stall", "open_air_stall", "mobile_hawker" (or null)
   - stock_value_kes: integer or null
   - monthly_budget_kes: integer or null
   - monthly_income_kes: integer or null
   - primary_risk_concern: "fire", "theft", "health", "accident", "weather_flood" (or null)
   - wants_cover_fire: boolean or null
   - wants_cover_theft: boolean or null
   - wants_cover_health: boolean or null
   - market_name: string or null
4. Provide Search Queries: 1 to 2 targeted English queries for ChromaDB retrieval.
5. Missing Essential Info: List any missing critical fields ("stall_type", "stock_value_kes", "primary_risk_concern") ONLY IF a recommendation is needed.

Output strict JSON:
{
  "detected_language": "en" | "sw" | "sheng" | "mixed",
  "routing_path": "info_only" | "recommendation_only" | "combined" | "clarification_reply" | "out_of_domain",
  "extracted_slots": {},
  "search_queries": [],
  "resolved_query": "Pronoun-resolved search query",
  "missing_slots_for_rec": []
}
"""

def route_and_extract(
    message: str,
    history: List[Dict[str, Any]],
    current_profile: Dict[str, Any]
) -> Dict[str, Any]:
    client = genai.Client(api_key=get_gemini_api_key())

    lines = []
    for turn in history[-4:]:
        role = "Trader" if turn["role"] == "user" else "ReCover"
        lines.append(f"{role}: {turn['text']}")
    hist_text = "\n".join(lines) if lines else "None"

    prompt = (
        ROUTER_PROMPT
        .replace("__HISTORY__", hist_text)
        .replace("__PROFILE__", json.dumps(current_profile))
        .replace("__MESSAGE__", message)
    )

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    try:
        return json.loads(response.text)
    except Exception:
        return {
            "detected_language": "en",
            "routing_path": "info_only",
            "extracted_slots": {},
            "search_queries": [message],
            "resolved_query": message,
            "missing_slots_for_rec": []
        }

print("Router & Extractor compiled.")


Router & Extractor compiled.


## 9. Grounded Conversational Generation & Citation Audit

Synthesizes responses using conversational Kenyan peer tone, strictly grounds figures/conditions against retrieved document excerpts, formats recommendation rationale, and verifies bracketed citations (`[S1]`, `[S2]`).


In [10]:
LOCALIZED_FALLBACKS = {
    "en": "I don't have verified information about that topic in the approved microinsurance documents. I am here to help explain insurance terms, claims steps, and policy rules.",
    "sw": "Sina maelezo yaliyothibitishwa kuhusu swali hilo kwenye nyaraka rasmi za bima. Niko hapa kukusaidia kuelewa maana ya bima, jinsi ya kulipa, na jinsi ya kudai fidia.",
    "sheng": "Sina hiyo info kwa ma-doc rasmi za bima, so sitaki kukupea story za jaba. Naweza kukusaidia kuelewa vile bima inawork, madeductibles, na vile unaclaim ukipata loss."
}

UNIFIED_SYSTEM_PROMPT = """You are ReCover, an empathetic, expert microinsurance advisor for informal market traders in Kenya (mitumba sellers, kiosk owners, food vendors).

Core Communication Rules:
1. Tone & Persona: Warm, grounded, peer-like, and empathetic. Naturally mirror the trader's language (English, Kiswahili, or Sheng).
2. Grounded Evidence: Rely strictly on provided excerpts for figures, waiting periods, deductibles, exclusions, and claims requirements. Cite them as [S1], [S2].
3. Realistic Guidance: When recommending products:
   - Name the underwriter and product explicitly.
   - Explain WHY it matches their stall type and budget.
   - Clarify that recommendations are consultative guidance, not an automated guarantee of coverage.
4. Missing Information: If essential details are missing for a recommendation, ask 1 or 2 polite, concise questions instead of guessing.
5. Everyday Analogies: Use market-relevant analogies (e.g. comparing risk pooling to a chama, or explaining deductibles using damaged clothing bales).
"""

def generate_unified_response(
    message: str,
    lang: str,
    routing_path: str,
    evidence: List[Dict[str, Any]],
    rec_results: Optional[Dict[str, Any]],
    missing_slots: List[str],
    history: List[Dict[str, Any]]
) -> str:
    if routing_path == "out_of_domain":
        return LOCALIZED_FALLBACKS.get(lang, LOCALIZED_FALLBACKS["en"])

    client = genai.Client(api_key=get_gemini_api_key())

    hist_text = "\n".join(
        f"{'Trader' if t['role'] == 'user' else 'ReCover'}: {t['text']}"
        for t in history[-4:]
    ) if history else "None"

    evidence_text = "\n\n".join(
        f"[S{i+1}] Source: {e['source']}, Page {e['page']}\n{e['text']}"
        for i, e in enumerate(evidence)
    ) if evidence else "No direct document excerpts retrieved."

    rec_summary = "None (No product recommendation was run for this turn)."
    if rec_results and rec_results.get("recommendations"):
        recs = rec_results["recommendations"]
        rec_summary = "\n".join(
            f"- {r['name']} (Verdict: {r['verdict']}): ~KES {r['monthly_cost_kes']}/month. Rationale: {r['reason']}"
            for r in recs
        )

    user_payload = f"""Conversation History:
{hist_text}

Retrieved Document Excerpts (Information System):
{evidence_text}

Recommendation System Output:
{rec_summary}

Missing Essential Parameters (if any):
{missing_slots}

Routing Path: {routing_path}
Detected Language: {lang}
Trader's Latest Message: "{message}"

Compose a unified, warm, and clear response for the trader."""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            {"role": "user", "parts": [{"text": UNIFIED_SYSTEM_PROMPT + "\n\n" + user_payload}]}
        ],
        config={"temperature": 0.25}
    )

    return response.text.strip()

def validate_and_extract_citations(answer: str, evidence: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    tags = re.findall(r"\[S(\d+)\]", answer)
    cited_indices = set(int(t) - 1 for t in tags)

    sources = []
    for idx in sorted(cited_indices):
        if 0 <= idx < len(evidence):
            chunk = evidence[idx]
            sources.append({
                "tag": f"[S{idx+1}]",
                "document": chunk["source"],
                "page": chunk["page"],
                "reference": f"{chunk['source']}, p.{chunk['page']}"
            })
    return sources

print("Generation and Citation validator compiled.")


Generation and Citation validator compiled.


## 10. Unified ReCoverChatbot Master Facade

Coordinates knowledge ingestion, vector indexing, multi-turn memory, NLP routing, dense retrieval, recommendation evaluation, and grounded synthesis into a single unified `bot.ask()` interface.


In [16]:
class ReCoverChatbot:
    def __init__(self):
        self.memory = ReCoverSessionMemory(max_turns=MAX_MEMORY_TURNS)
        self.indexer = VectorIndexer()
        self._initialize_knowledge_base()

    def _initialize_knowledge_base(self) -> None:
        ensure_educational_primer()
        all_pages = []

        for file_path in DOCS_DIR.iterdir():
            if file_path.suffix.lower() == ".txt":
                all_pages.extend(split_ocr_pages(file_path, file_path.name))
            elif file_path.suffix.lower() == ".pdf":
                resolved = run_gemini_ocr_if_needed(file_path)
                if resolved.suffix.lower() == ".txt":
                    all_pages.extend(split_ocr_pages(resolved, file_path.name))
                else:
                    all_pages.extend(extract_pdf_pages(file_path))

        chunks = chunk_pages(all_pages)
        self.indexer.build_index(chunks)

    def ask(self, user_message: str) -> Dict[str, Any]:
        cleaned = user_message.strip()
        if not cleaned:
            return {
                "answer": "Habari! Ask any microinsurance question or tell me about your business to get started.",
                "sources": [],
                "routing_path": "info_only",
                "detected_language": "en",
                "trader_profile": self.memory.get_profile(),
                "recommendations": []
            }

        history = self.memory.get_history()
        current_profile = self.memory.get_profile()

        # 1. NLP Routing & Slot Extraction
        routing_data = route_and_extract(cleaned, history, current_profile)
        self.memory.update_profile(routing_data.get("extracted_slots", {}))
        updated_profile = self.memory.get_profile()

        routing_path = routing_data.get("routing_path", "info_only")
        lang = routing_data.get("detected_language", "en")

        # 2. Information Retrieval (RAG)
        evidence = []
        if routing_path in ["info_only", "combined", "clarification_reply"]:
            queries = routing_data.get("search_queries", [cleaned])
            evidence = retrieve_evidence(queries, self.indexer)

        # 3. Recommendation Evaluation
        rec_results = None
        missing_slots = routing_data.get("missing_slots_for_rec", [])
        if routing_path in ["recommendation_only", "combined"]:
            has_min_info = updated_profile.get("stall_type") and updated_profile.get("stock_value_kes")
            if has_min_info:
                rec_results = evaluate_product_fit(updated_profile)
                missing_slots = []

        # 4. Response Generation
        answer = generate_unified_response(
            message=cleaned,
            lang=lang,
            routing_path=routing_path,
            evidence=evidence,
            rec_results=rec_results,
            missing_slots=missing_slots,
            history=history
        )

        # 5. Extract & Merge Citations (Educational RAG + Recommendation Policy Provenance)
        sources = validate_and_extract_citations(answer, evidence)

        if rec_results and rec_results.get("recommendations"):
            for rec in rec_results["recommendations"]:
                if rec.get("verdict") in ["may_qualify", "needs_confirmation"]:
                    sources.append({
                        "tag": "[Rec]",
                        "document": rec["source_doc"],
                        "page": rec["source_page"],
                        "reference": f"{rec['source_doc']}, p.{rec['source_page']} ({rec['source_clause']})"
                    })

        # Deduplicate sources while preserving order
        unique_sources = []
        seen = set()
        for s in sources:
            if s["reference"] not in seen:
                unique_sources.append(s)
                seen.add(s["reference"])

        # 6. Commit Turn to Session Memory
        self.memory.add_user_message(cleaned)
        self.memory.add_assistant_response(answer, [s["reference"] for s in unique_sources])

        return {
            "answer": answer,
            "sources": unique_sources,
            "routing_path": routing_path,
            "detected_language": lang,
            "trader_profile": updated_profile,
            "recommendations": rec_results["recommendations"] if rec_results else []
        }

    def reset(self) -> None:
        self.memory.clear()

print("ReCoverChatbot Facade successfully initialized with full provenance.")

ReCoverChatbot Facade successfully initialized with full provenance.


## 11. Multi-Scenario Validation Test Suite

Validates the integrated chatbot across all required test matrices:
- **Test 1:** Information-Only (Path 1)
- **Test 2:** Multi-Turn Recommendation with Incomplete Profile & Clarification (Path 2)
- **Test 3:** Combined Education & Recommendation with Sheng Code-Switching (Path 3)
- **Test 4:** Out-of-Domain Safety Guardrail


In [12]:
# Initialize bot instance
recover_bot = ReCoverChatbot()

def run_test(scenario_label: str, message: str):
    print("=" * 70)
    print(f"SCENARIO: {scenario_label}")
    print(f"Trader Input: '{message}'")
    print("-" * 70)
    res = recover_bot.ask(message)
    print(f"Path: {res['routing_path']} | Lang: {res['detected_language']}")
    print(f"Profile: {res['trader_profile']}")
    print(f"ReCover Response:\n{res['answer']}")
    if res['sources']:
        print("\nVerified Sources:")
        for s in res['sources']:
            print(f"  • {s['tag']} {s['reference']}")
    print("=" * 70 + "\n")

# Execute Test 1: Information-Only
run_test("Test 1: Information-Only (Deductible Explanation)", "What is a deductible and why does an insurer make me pay it?")

# Execute Test 2A: Recommendation with Missing Information
run_test("Test 2A: Recommendation Missing Info", "Which insurance policy should I get for my shop?")

# Execute Test 2B: Conversational Slot-Filling Follow-up
run_test("Test 2B: Slot Clarification Reply", "I sell shoes in a permanent shop at Toi Market, my stock is around KES 150,000 and budget is KES 500 a month.")

# Execute Test 3: Combined Information & Recommendation in Sheng
run_test("Test 3: Combined Info & Rec (Sheng Code-Switch)", "Kibanda yangu ya mitumba Toi ilichomeka. Nataka kujua kama bima itanilipa nikichukua, na ni cover gani naeza pata ya 80k stock?")

# Execute Test 4: Out of Domain
run_test("Test 4: Out of Domain Guardrail", "Who won the English Premier League football match yesterday?")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexing 3 chunks into ChromaDB...
Successfully indexed 3 chunks.
SCENARIO: Test 1: Information-Only (Deductible Explanation)
Trader Input: 'What is a deductible and why does an insurer make me pay it?'
----------------------------------------------------------------------
Path: info_only | Lang: en
Profile: {'stall_type': None, 'stock_value_kes': None, 'monthly_budget_kes': None, 'monthly_income_kes': None, 'primary_risk_concern': None, 'wants_cover_fire': None, 'wants_cover_theft': None, 'wants_cover_health': None, 'market_name': None}
ReCover Response:
Habari! That is one of the most common and important questions traders ask, so let’s break it down simply.

In insurance, a **deductible**—also commonly called an **excess** here in Kenya—is simply the first agreed portion of any claim that you pay out of pocket before the insurance company covers the remaining balance [S1]. 

### A Quick Market Analogy
Think of it like opening a bale of mitumba clothes. If just one or two shirts have

## 12. Interactive Chat Session

Run the cell below to engage in an interactive chat session with ReCover. Type `exit` to stop or `reset` to start a fresh trader session.


In [17]:
def run_interactive_session():
    bot = ReCoverChatbot()
    print("=" * 60)
    print("ReCover Interactive Session Started")
    print("Type 'exit' to quit, 'reset' to clear conversation memory.")
    print("=" * 60)

    while True:
        try:
            msg = input("\nTrader: ").strip()
            if not msg:
                continue
            if msg.lower() == "exit":
                print("Session terminated.")
                break
            if msg.lower() == "reset":
                bot.reset()
                print("Session memory and business profile cleared.")
                continue

            response = bot.ask(msg)
            print(f"\nReCover:\n{response['answer']}")
            if response["sources"]:
                print("\nSources:")
                for s in response["sources"]:
                    print(f"  • {s['tag']} {s['reference']}")
        except KeyboardInterrupt:
            print("\nSession exited.")
            break


run_interactive_session()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection 'recover_integrated_educational_docs' already populated with 3 chunks.
ReCover Interactive Session Started
Type 'exit' to quit, 'reset' to clear conversation memory.

Trader: Hello, nikona stock ya 150k, insurance ipi inanifaa?

ReCover:
Sasa boss! Hongera sana kwa kazi—stock ya 150k ni jasho na mtaji mzito sana, na ni hatua fiti sana unafikiria kuilinda mapema ndio hustle yako isiyumbe.

Ili nikusaidie kupata cover inayokufaa vizuri na isiyoumiza mfuko, ningependa kukuuliza maswali mawili madogo tu:

1. **Unauza nini na uko base ya aina gani?** (Mfano: mitumba soko wazi, kiosk ya mbao/mabati, kibanda ya mboga, ama duka ya permanent?)
2. **Ni hatari gani haswa inakupea wasiwasi zaidi kwa hii stock?** (Mfano: moto, wizi, mvua/mafuriko, au ukipata emergency ya afya ikusimamishe kazi?)

Ukinidondoshea hizi details, nitaweza kukuongoza kwa cover spesheli itakayochunga mali yako vizuri!

Trader: soko wazi, naogopa sana moto na shida za health unajua tu hizi mambo hutokea ghafla

